In [1]:
import glob
import os
import re
from typing import List, Optional

import pandas as pd
import plotly.graph_objects as go

In [2]:
def find_mask_files(pattern: str) -> List[str]:
    """Находит файлы по заданному паттерну."""
    files = glob.glob(pattern)
    files.sort()
    return files

In [3]:
def read_tecplot_ascii(file_path: str) -> pd.DataFrame:
    """
    Читает ASCII формат Tecplot.
    Пропускает заголовок и читает данные в DataFrame.
    """
    # Tecplot ASCII имеет заголовок переменной длины.
    # В твоем примере это 3 строки, но надежнее найти начало данных.
    # Обычно данные начинаются после строки ZONE.
    
    header_rows = 0
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            line = line.strip()
            # Простая эвристика: если строка начинается с цифры, это данные
            if line and (line[0].isdigit() or line[0] == '-'):
                header_rows = i
                break
    
    # Читаем данные. engine='c' быстрее. sep='\s+' обрабатывает любые пробелы
    df = pd.read_csv(
        file_path,
        skiprows=header_rows,
        sep=r'\s+',
        names=['X', 'Y', 'Z', 'mask'],
        engine='python' # python engine нужен для regex separator, c engine быстрее, но строже
    )
    return df

In [5]:
def read_tecplot_ascii_LAD(file_path: str) -> pd.DataFrame:
    """
    Читает ASCII формат Tecplot для LAD.
    Пропускает заголовок и читает данные в DataFrame.
    """
    # Tecplot ASCII имеет заголовок переменной длины.
    # В твоем примере это 3 строки, но надежнее найти начало данных.
    # Обычно данные начинаются после строки ZONE.
    
    header_rows = 0
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            line = line.strip()
            # Простая эвристика: если строка начинается с цифры, это данные
            if line and (line[0].isdigit() or line[0] == '-'):
                header_rows = i
                break
    
    # Читаем данные. engine='c' быстрее. sep='\s+' обрабатывает любые пробелы
    df = pd.read_csv(
        file_path,
        skiprows=header_rows,
        sep=r'\s+',
        names=['X', 'Y', 'Z', 'LAD'],
        engine='python' # python engine нужен для regex separator, c engine быстрее, но строже
    )
    return df

In [6]:
def plot_3d_mask(file_path: str, output_html: Optional[str] = None):
    """
    Визуализирует точки маски, где значение > 0.
    """
    print(f"Processing: {file_path}")
    
    try:
        df = read_tecplot_ascii(file_path)
    except Exception as e:
        print(f"Error reading file: {e}")
        return

    # Оптимизация: берем только точки, входящие в маску
    # Предполагаем, что mask - это бинарное (0/1) или float поле.
    mask_points = df[df['mask'] != 0.0]
    # mask_points = mask_points[mask_points['mask'] > 0.0]


    if mask_points.empty:
        print("Warning: Mask is empty (no points > 0.5 found).")
        return

    print(f"Found {len(mask_points)} active points out of {len(df)} total.")

    fig = go.Figure(data=[go.Scatter3d(
        x=mask_points['X'],
        y=mask_points['Y'],
        z=mask_points['Z'],
        mode='markers',
        marker=dict(
            size=3,              # Размер точек
            color=mask_points['Z'], # Раскрасим по высоте (Z) для наглядности
            colorscale='Viridis',
            opacity=0.8
        )
    )])

    # Настройка сцены для правильных пропорций
    fig.update_layout(
        title=f"3D Mask Visualization: {os.path.basename(file_path)}",
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data' # Важно, чтобы не искажались пропорции маски
        ),
        margin=dict(r=0, l=0, b=0, t=40)
    )

    if output_html:
        fig.write_html(output_html)
        print(f"Saved to {output_html}")
    else:
        fig.show()

## draw mask

In [8]:
search_path = '/app/urban-layer/build/output_2025_12_29_14_44_8/common/3d/mask-.plt'

files = find_mask_files(search_path)

if not files:
    print(f"No files found for pattern: {search_path}")

print(f"Found {len(files)} files.")

# Берем первый попавшийся файл для примера
target_file = files[0]

# Отображаем
plot_3d_mask(target_file)

Found 1 files.
Processing: /app/urban-layer/build/output_2025_12_29_14_44_8/common/3d/mask-.plt
Found 8192 active points out of 524288 total.


## draw LAD

In [17]:
search_path = '/app/urban-layer/build_cuda1222_scale/output_2026_1_11_14_14_18/common/3d/LAD-.plt'

files = find_mask_files(search_path)

if not files:
    print(f"No files found for pattern: {search_path}")

print(f"Found {len(files)} files.")

target_file = files[0]
plot_3d_mask(target_file)

Found 1 files.
Processing: /app/urban-layer/build_cuda1222_scale/output_2026_1_11_14_14_18/common/3d/LAD-.plt
Found 12857 active points out of 524288 total.


In [13]:
df = read_tecplot_ascii_LAD(target_file)

In [14]:
df_un = df[df['LAD'] > 0]

In [15]:
df_un.shape

(0, 4)